# MEDIC SPI training campaign
Image-only four-class pseudo-label agreement. Separate post-text targets where available. Four frozen encoders, nine vector-head families, attention heads and mean-pool controls. Kaggle managed Python; setup prints exact environment and source SHA. Test locked until model selection freezes. Training outputs remain research candidates; no automatic promotion. Artifacts/logs are incremental. No full dataset archives.

In [ ]:
import os,sys,time,json,platform,subprocess,hashlib,base64,io,zipfile,traceback
from pathlib import Path
print('SETUP START',sys.executable,sys.version,flush=True)
PROJECT=Path('/kaggle/working/project');PROJECT.mkdir(exist_ok=True)
RUN=Path('/kaggle/working/experiments/spi_classifier_repair/runs')/('medic_spi_'+time.strftime('%Y%m%dT%H%M%SZ',time.gmtime()));RUN.mkdir(parents=True)
os.environ['YOLO_CONFIG_DIR']=str(RUN/'yolo_config');os.environ['OMP_NUM_THREADS']='1';os.environ['MKL_NUM_THREADS']='1'
class Tee:
 def __init__(self,stream,path): self.stream=stream;self.file=open(path,'a',buffering=1)
 def write(self,s): self.stream.write(s);self.file.write(s);return len(s)
 def flush(self): self.stream.flush();self.file.flush()
sys.stdout=Tee(sys.stdout,RUN/'stdout.log');sys.stderr=Tee(sys.stderr,RUN/'stderr.log')
blob=base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest()=='2142d9738d7e2fc7b785c50519c567cdb4cff136602395e91f8326a85fd79c2d'
with zipfile.ZipFile(io.BytesIO(blob)) as z:z.extractall(PROJECT)
sys.path.insert(0,str(PROJECT));os.environ['PYTHONPATH']=str(PROJECT)
print('SOURCE VERIFIED',hashlib.sha256(blob).hexdigest(),flush=True)
import torch
print('HARDWARE',torch.__version__,torch.cuda.device_count(),[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)
assert torch.cuda.device_count()==2,'two GPUs required'
commands=[[sys.executable,'-m','pip','install','transformers==5.5.0','huggingface-hub==1.5.0','sentencepiece==0.2.2','ultralytics==8.3.237','ftfy','regex','requests','nbformat==5.10.4','bitsandbytes==0.49.2','accelerate==1.12.0','scikit-learn==1.7.2','joblib','threadpoolctl','psutil','matplotlib'],[sys.executable,'-m','pip','install','--no-deps','--no-build-isolation','git+https://github.com/openai/CLIP.git@a1d071733d7111c9c014f024669f959182114e33']]
for i,command in enumerate(commands):
 print('DEPENDENCIES START',i,flush=True)
 with (RUN/f'install_{i}.log').open('w') as log:subprocess.run(command,stdout=log,stderr=subprocess.STDOUT,check=True,timeout=300)
 print('DEPENDENCIES END',i,flush=True)
import importlib.metadata as md
(RUN/'environment.json').write_text(json.dumps({'python':sys.version,'torch':torch.__version__,'packages':{k:md.version(k) for k in ['transformers','ultralytics','numpy','huggingface-hub']},'source_sha256':hashlib.sha256(blob).hexdigest()},indent=2))
print('SETUP END',RUN,flush=True)


In [ ]:
print('EXACT MEDIC COHORT LOAD START',flush=True)
from parameters.artifacts import write_json
from experiments.spi_classifier_repair.resume_store import file_hash
found=[]
for path in Path('/kaggle/input').glob('**/input_manifest.json'):
 try:
  manifest=json.loads(path.read_text())
  if manifest.get('schema')=='medic300_v18':found.append((path,manifest))
 except (ValueError,OSError):continue
if len(found)!=1:raise RuntimeError('Attach exactly one verified MEDIC300 snapshot; no automatic reacquisition or resampling')
manifest_path,manifest=found[0];input_root=manifest_path.parent
records_path=input_root/'records.json'
expected='1b0feb7ec30c3d9fb033d9df8456daf37e56b5941f35e40aeb684d02e98845ae'
if file_hash(records_path)!=expected or manifest.get('original_records_sha256')!=expected:raise RuntimeError('MEDIC cohort manifest differs')
rows=json.loads(records_path.read_text());schema=json.loads((PROJECT/'parameters/config/medic_native_schema_v18.json').read_text())['features']
for row in rows:
 matches=list(input_root.glob('**/'+row['image_sha256']+'.img'))
 if len(matches)!=1:raise RuntimeError('exact upstream image missing')
 if file_hash(matches[0])!=row['image_sha256']:raise RuntimeError('MEDIC image hash mismatch')
 row['image_path']=str(matches[0]);row['prior_detector_study_exposure']=True
 row['native_features']=schema
if len(rows)!=300 or any(r['source']!='medic' for r in rows):raise RuntimeError('MEDIC300 cohort mismatch')
write_json(RUN/'data/records.json',rows)
write_json(RUN/'input_receipt.json',{'manifest':manifest,'verified_images':300,'records_sha256':expected,'source_sha256':hashlib.sha256(blob).hexdigest()})
# Restore only an explicitly identified compatible checkpoint archive, if attached.
for index_path in Path('/kaggle/input').glob('**/resume_index.json'):
 resume=json.loads(index_path.read_text())
 if resume.get('schema')!='medic_spi_checkpoints_v1' or resume.get('records_sha256')!=expected:continue
 archive=index_path.parent/resume['archive']
 if file_hash(archive)!=resume['sha256']:raise RuntimeError('resume archive checksum mismatch')
 with zipfile.ZipFile(archive) as saved:
  for item in saved.infolist():
   target=(RUN/item.filename).resolve()
   if not target.is_relative_to(RUN.resolve()) or not item.filename.startswith('campaign/'):raise RuntimeError('unsafe resume archive')
  saved.extractall(RUN)
 print('RESTORED CHECKPOINTS',archive,flush=True)
print('EXACT MEDIC COHORT LOAD END',len(rows),flush=True)


In [ ]:
print('SPI LABEL/TRAIN/PROFILE START',flush=True)
from experiments.spi_classifier_repair.medic_campaign import run
started=time.monotonic()
try:
 run(RUN/'data/records.json',PROJECT/'parameters/results/spi_inputs_v2_20261027b/panel.json',PROJECT/'parameters/config/final_labeling_teacher.json',RUN/'campaign')
except BaseException:
 (RUN/'failure.txt').write_text(traceback.format_exc());raise
finally:
 print('CAMPAIGN ELAPSED',time.monotonic()-started,flush=True)
 # Checkpoints persist incrementally; package them even on a stage exception.
 print('RECOVERY PACKAGING START',flush=True)
 recovered=[]
 with zipfile.ZipFile('/kaggle/working/spi_recovery.zip','w',zipfile.ZIP_STORED) as archive:
  for file in RUN.rglob('*'):
   if file.is_file():
    name=file.relative_to(RUN).as_posix();archive.write(file,name);recovered.append({'path':name,'bytes':file.stat().st_size,'sha256':file_hash(file)})
 (Path('/kaggle/working')/'spi_recovery_manifest.json').write_text(json.dumps({'run':str(RUN),'files':recovered},indent=2))
 print('RECOVERY PACKAGING END',len(recovered),flush=True)
